# 02 — Feature Engineering
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Create temporal and monetary features
- Handle class imbalance with SMOTE
- Produce train/test splits ready for DB2 ingestion and AutoAI

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from src.feature_engineering import TimeFeatures, AmountFeatures
import warnings
warnings.filterwarnings('ignore')

## 1. Load Sampled Data

In [ ]:
df = pd.read_csv('../data/raw/transactions_sampled.csv')
print(f'Shape: {df.shape} | Fraud rate: {df["Class"].mean():.4%}')
df.head(3)

## 2. Feature Engineering

In [ ]:
# Apply transformers
df = TimeFeatures().fit_transform(df)
df = AmountFeatures().fit_transform(df)

# Encode categorical features
df = pd.get_dummies(df, columns=['time_bucket', 'amount_bucket'], drop_first=True)

# Drop raw columns already encoded
df = df.drop(columns=['Time', 'Amount'])

print('New features added:')
new_cols = [c for c in df.columns if c not in [f'V{i}' for i in range(1,29)] + ['Class', 'hour', 'log_amount']]
print(new_cols)
print(f'\nFinal shape: {df.shape}')
df.head(3)

In [ ]:
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After  SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(
    axes,
    [y_train.value_counts(), pd.Series(y_train_res).value_counts()],
    ['Antes do SMOTE', 'Após SMOTE']
):
    ax.bar(['Legítima', 'Fraude'], counts.values, color=['#0062ff', '#ff5c49'])
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns='Class')
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape} | Fraud rate: {y_train.mean():.4%}')
print(f'Test:  {X_test.shape}  | Fraud rate: {y_test.mean():.4%}')

## 4. SMOTE — Balancing the Training Set
SMOTE (Synthetic Minority Oversampling Technique) generates synthetic fraud cases  
in the feature space, without simply repeating existing samples.

In [ ]:
smote = SMOTE(sampling_strategy=0.1, random_state=42, n_jobs=-1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After  SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(
    axes,
    [y_train.value_counts(), pd.Series(y_train_res).value_counts()],
    ['Antes do SMOTE', 'Após SMOTE']
):
    ax.bar(['Legítima', 'Fraude'], counts.values, color=['#0062ff', '#ff5c49'])
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Save Processed Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Full dataset for DB2 ingestion (original split, no SMOTE — DB2 stores raw features)
train_df = X_train.copy()
train_df['Class'] = y_train.values
train_df['split'] = 'train'

test_df = X_test.copy()
test_df['Class'] = y_test.values
test_df['split'] = 'test'

full_df = pd.concat([train_df, test_df]).reset_index(drop=True)
full_df.to_csv('../data/processed/transactions_features.csv', index=False)

# Resampled training set for local baseline
X_train_res_df = pd.DataFrame(X_train_res, columns=X_train.columns)
X_train_res_df['Class'] = y_train_res
X_train_res_df.to_csv('../data/processed/train_resampled.csv', index=False)

print(f'transactions_features.csv: {len(full_df):,} rows, {full_df.shape[1]} cols')
print(f'train_resampled.csv:       {len(X_train_res_df):,} rows')
print(f'Features list: {list(X_train.columns)}')